# Pytorch의 nn.Embedding
- Pytorch의 Embedding Layer는 word2vec과 마찬가지로 word embedding vector를 찾는 **Lookup Table**이다.
    - 단어의 **정수의 고유 index**가 입력으로 들어오면 Embedding Layer의 **그 index의 Vector**를 출력한다.
    - 모델이 학습되는 동안 모델이 풀려는 문제에 맞는 값으로 Embedding Layer의 vector들이 업데이트 된다.
    - Word2Vec의 embedding vector 학습을 nn.Embedding은 자신이 포함된 모델을 학습 하는 과정에서 한다고 생각하면 된다.

In [8]:
import torch
import torch.nn as nn

embedding_layer = nn.Embedding(
    # vocab size(총 어휘수). 20_000개 단어(토큰) 에 대한 embedding vector를 만들겠다.
    num_embeddings=20_000, 
    embedding_dim=100, # embedding vector의 차원수. 하나의 토큰을 몇개의 숫자로 분산표현할 것인지.
    padding_idx=0 # Padding 토큰(<PAD>)의 어휘사전에서의 index. -> 학습하지 않는다.
)

# 20_000 x 100 짜리 Embedding Vector 를 생성.

In [9]:
w = embedding_layer.weight
print(w.shape)

torch.Size([20000, 100])


In [10]:
# 3번 토큰 단어의 embedding vector를 조회
w[2]

tensor([ 0.0998,  0.6687, -0.6133,  0.1169, -0.1407, -0.0138, -0.9171,  0.8540,
         0.1124,  0.5657, -0.5783, -0.6725, -0.9996,  1.3427,  0.3616,  0.7940,
         1.1563,  1.0909, -0.3481,  0.6170,  1.5109,  0.2154,  0.8004,  0.3304,
        -0.4817, -0.2292,  0.2128, -0.1004,  0.1877, -1.3929, -0.0275, -0.2941,
         0.9004, -0.7708,  0.0213, -0.6330,  0.4382, -0.0658, -0.4250, -0.0910,
         1.7829, -0.7550,  0.6243, -1.2402,  0.5836, -0.2830,  1.1675, -1.1724,
        -1.0567,  1.5582,  1.1339,  0.5027, -0.7021, -2.6157, -1.1602, -1.3124,
        -1.8562, -0.4870, -0.0598,  0.4036,  0.6896,  0.1333,  1.8026, -1.5251,
         0.8387, -0.5861, -2.2052, -0.3510, -1.2622, -0.6286, -2.0222, -0.7320,
         0.6143, -0.3461,  1.5480, -0.2695,  1.0955,  0.1547,  0.2400, -1.2775,
         0.5777, -0.5891,  0.7666,  0.6897,  0.8419, -0.2045, -1.2381,  0.6338,
        -0.5879, -0.2339, -0.0493,  0.0765, -0.6294, -2.0801,  1.1787,  1.0412,
         1.3817, -0.1260,  0.3866, -0.21

In [11]:
# 0번 토큰 - <pad>
w[0]

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], grad_fn=<SelectBackward0>)

# 네이버 영화 댓글 감성분석(Sentiment Analysis)

## 감성분석(Sentiment Analysis) 이란
입력된 텍스트가 **긍적적인 글**인지 **부정적인**인지 또는 **중립적인** 글인지 분석하는 것을 감성(감정) 분석이라고 한다.   
이를 통해 기업이 고객이 자신들의 기업 또는 제품에 대해 어떤 의견을 가지고 있는지 분석한다.

# Dataset, DataLoader 생성

## Korpora에서 Naver 영화 댓글 dataset 가져오기
- Korpora: https://github.com/ko-nlp/Korpora
- NSMC: http://github.com/e9t/nsmc/
    - input: 영화댓글
    - output: 0(부정적댓글), 1(긍정적댓글)
### API
- **corpus 가져오기**
    - `Korpora.load('nsmc')`
- **text/label 조회**
    - `corpus.get_all_texts()` : 전체 corpus의 text들을 tuple로 반환
    - `corpus.get_all_labels()`: 전체 corpus의 label들을 list로 반환
- **train/test set 나눠서 조회**
    - `corpus.train`
    - `corpus.test`
    - `LabeledSentenceKorpusData` 객체에 text와 label들을 담아서 제공.
        - `LabeledSentenceKorpusData.texts`: text들 tuple로 반환.
        - `LabeledSentenceKorpusData.labels`: label들 list로 반환.

## 데이터 로딩

In [3]:
from Korpora import Korpora
corpus = Korpora.load("nsmc")


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at C:\Users\Playdata\Korpora\nsmc\ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at C:\Users\

In [4]:
all_text = corpus.get_all_texts()
all_label = corpus.get_all_labels() 

all_text[:5]

('아 더빙.. 진짜 짜증나네요 목소리',
 '흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나',
 '너무재밓었다그래서보는것을추천한다',
 '교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정',
 '사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 던스트가 너무나도 이뻐보였다')

In [ ]:
all_label[:5] # 0: 부정적, 1: 긍정적

[0, 1, 0, 0, 1]

In [16]:
len(all_text)

200000

In [17]:
# train/test set 조회
corpus.train

NSMC.train: size=150000
  - NSMC.train.texts : list[str]
  - NSMC.train.labels : list[int]

In [19]:
corpus.test.texts

('굳 ㅋ',
 'GDNTOPCLASSINTHECLUB',
 '뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아',
 '지루하지는 않은데 완전 막장임... 돈주고 보기에는....',
 '3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??',
 '음악이 주가 된, 최고의 음악영화',
 '진정한 쓰레기',
 '마치 미국애니에서 튀어나온듯한 창의력없는 로봇디자인부터가,고개를 젖게한다',
 '갈수록 개판되가는 중국영화 유치하고 내용없음 폼잡다 끝남 말도안되는 무기에 유치한cg남무 아 그립다 동사서독같은 영화가 이건 3류아류작이다',
 '이별의 아픔뒤에 찾아오는 새로운 인연의 기쁨 But, 모든 사람이 그렇지는 않네..',
 '괜찮네요오랜만포켓몬스터잼밌어요',
 '한국독립영화의 한계 그렇게 아버지가 된다와 비교됨',
 '청춘은 아름답다 그 아름다움은 이성을 흔들어 놓는다. 찰나의 아름다움을 잘 포착한 섬세하고 아름다운 수채화같은 퀴어영화이다.',
 '눈에 보이는 반전이었지만 영화의 흡인력은 사라지지 않았다.',
 '"""스토리, 연출, 연기, 비주얼 등 영화의 기본 조차 안된 영화에 무슨 평을 해. 이런 영화 찍고도 김문옥 감독은 """"내가 영화 경력이 몇OO인데 조무래기들이 내 영화를 평론해?"""" 같은 마인드에 빠져있겠지?"""',
 '소위 ㅈ문가라는 평점은 뭐냐?',
 '최고!!!!!!!!!!!!!!!!',
 '발연기 도저히 못보겠다 진짜 이렇게 연기를 못할거라곤 상상도 못했네',
 '나이스',
 '별 재미도없는거 우려먹어 .... 챔프에서 방송 몇번했더라 ? ㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋㅋ',
 "'13일의 금요일', '나이트메어'시리즈와 함께 가장 많은 시리즈를 양산해냈던 헬레이저 시리즈의 첫편. 작가의 상상력이 돋보이는 작품이며, 갈고리로 사지찢는 고어씬은 지금보더라도 상당히 잔인하고 충격적이다.",
 '나름 교훈돋기는 하지만 어쩔수없이 저평점 받을수밖에 없는 저질섹스코미디',
 '꽤 재밌게 본 영화였다!',

## 토큰화
1. 형태소 단위 token화
    - konlpy로 token화 한 뒤 다시 한 문장으로 만든다.
2. 1에서 처리한 corpus를 BPE 로 token화
   
### 전처리 함수

#### 형태소 단위 분절

In [5]:
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    # 1.
    text = text.lower()
    # 2. 특수문자 -> 공백
    text = re.sub(rf"[{string.punctuation}]", " ", text)
    # 3. 
    text = [token.lemma for token in kiwi.tokenize(text)]
    # 4. 
    return " ".join(text)

In [6]:
print(all_text[20])
text_preprocessing(all_text[20])

나름 심오한 뜻도 있는 듯. 그냥 학생이 선생과 놀아나는 영화는 절대 아님


'나름 심오 하 ᆫ 뜻 도 있다 는 듯 그냥 학생 이 선생 과 놀아나다 는 영화 는 절대 아니다 ᆷ'

In [7]:
import string 
string.punctuation

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~'

In [ ]:
### 전처리
train_inputs = [text_preprocessing(txt) for txt in corpus.train.texts] # trainset 댓글-전처리
test_inputs = [text_preprocessing(txt) for txt in corpus.test.texts] # testset 댓글-전처리

train_labels = corpus.train.labels
test_labels = corpus.test.labels

In [14]:
# 피클로 저장
import os
os.makedirs("data/nsmc", exist_ok=True)

train_data = {"text": train_inputs, "label": train_labels}
test_data = {"text": test_inputs, "label": test_labels}

import pickle 
with open("data/nsmc/preprocessing_train.pkl", 'wb') as fo:
    pickle.dump(train_data, fo)

with open("data/nsmc/preprocessing_test.pkl", 'wb') as fo:
    pickle.dump(test_data, fo)

In [38]:
# 토큰화를 위해서 text 병합
all_inputs = train_inputs + test_inputs

### 토큰화
- Subword 방식 토큰화 적용
- Byte Pair Encoding 방식으로 huggingface tokenizer 사용
    - BPE: 토큰을 글자 단위로 나눈뒤 가장 자주 등장하는 글자 쌍(byte paire)를 찾아 합친뒤 어휘사전에 추가한다.
    - https://huggingface.co/docs/tokenizers/quicktour
    - `pip install tokenizers`

In [39]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer

vocab_size = 30_000

tokenizer = Tokenizer(
    BPE(unk_token="<unk>")
)
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=5,
    special_tokens=["<unk>", "<pad>"], # id: 0-<unk>, 1-<pad>
    continuing_subword_prefix="##"
)

# 학습데이터가 list[doc]
tokenizer.train_from_iterator(all_inputs, trainer=trainer)
# 학습데이터가 파일: tokenizer.train([파일경로], ..)

In [40]:
# 저장
os.makedirs("saved_model/nsmc/lstm", exist_ok=True)
tokenizer.save("saved_model/nsmc/lstm/bpe_tokenizer.json")
# Tokenizer.from_file(경로) - 로드

In [41]:
# 총 어휘수
tokenizer.get_vocab_size()

24855

In [42]:
idx = 120
print(all_inputs[idx])
encoding = tokenizer.encode(all_inputs[idx])
# encoding
print("토큰 ID")
print(encoding.ids)
print("토큰 문자열")
print(encoding.tokens)


중국인 특유 의 과장 허풍 있다 어 보이다 려고 안간힘 쓰다 ᆫ 노력 은 가상 하 나 고증 과 현실감 떨어지다 는 설정 이 거북 스럽 다 도대체 그 들 은 왜 이렇다 게 까지 스스로 를 과대 포장 하 는 것 이다 ᆫ지
토큰 ID
[9204, 6157, 2149, 6684, 17566, 5257, 1978, 5342, 5540, 1937, 3086, 3818, 5374, 59, 6094, 2135, 9046, 2815, 810, 8023, 622, 7397, 5575, 920, 5653, 2152, 7238, 5366, 946, 5643, 670, 1059, 2135, 2065, 5349, 580, 5313, 6771, 1284, 7760, 6372, 2815, 920, 574, 5251, 5332]
토큰 문자열
['중국인', '특유', '의', '과장', '허풍', '있다', '어', '보이다', '려고', '안', '##간', '##힘', '쓰다', 'ᆫ', '노력', '은', '가상', '하', '나', '고증', '과', '현실감', '떨어지다', '는', '설정', '이', '거북', '스럽', '다', '도대체', '그', '들', '은', '왜', '이렇다', '게', '까지', '스스로', '를', '과대', '포장', '하', '는', '것', '이다', 'ᆫ지']


## Dataset, DataLoader 생성

In [50]:
import torch
from torch.utils.data import Dataset, DataLoader

class NSMCDataset(Dataset):
    def __init__(self, texts, labels, max_length, tokenizer):
        """
        texts: list - 댓글 리스트. 리스트에 댓글들을 담아서 받는다. ["댓글", "댓글", ...]
        labels: list - Label 리스트. (댓글의 긍부정 여부 - 긍정: 1, 부정: 0)
        max_length: int: 개별 댓글의 최대 token 개수. 모든 댓글의 토큰수를 max_length에 맞춘다.
        tokenizer: tokenizers.Tokenizer
        """
        self.max_length = max_length 
        self.tokenizer = tokenizer
        self.labels = labels
        # text는 토큰화해서 저장.
        self.texts = [
            self.__pad_token_sequences(self.tokenizer.encode(txt).ids) for txt in texts
        ]

    ###########################################################################################
    # id로 구성된 개별 문장 token list를 받아서 패딩 추가 [20, 2, 1] => [20, 2, 1, 0, 0, 0, ..]
    ############################################################################################
    def __pad_token_sequences(self, token_sequences):
        """
        token id로 구성된 개별 문서(댓글)의 token_id list를 받아서 max_length 길이에 맞추는 메소드
        max_length 보다 토큰수가 적으면 Padding 토큰 추가, 많으면 max_length 크기로 줄인다.
            ex) max_length=5 이고 pad토큰 id가 0이라면
                [20, 2, 1] => [20, 2, 1, 0, 0]
                [20, 21, 30, 34, 60, 17, 21, 33] -> [20, 21, 30, 34, 60]
        """
        # 1. <pad> 토큰 id를 조회
        pad_token_id = self.tokenizer.token_to_id("<pad>")

        # 2. 입력 문서의 토큰 수
        seq_len = len(token_sequences)

        # 3. max_length 길이에 토큰수를 맞추기.
        if self.max_length < seq_len: # 많은 경우
            result = token_sequences[:self.max_length]
        else: # 모자란 경우
            result = token_sequences + ([pad_token_id] * (self.max_length - seq_len))

        return result


        
    def __len__(self):
        """총 데이터 수를 반환. len(dataset) 호출시 반환값"""
        return len(self.labels)

    def __getitem__(self, idx):
        """
        idx 번째 text와 label을 학습 가능한 type으로 변환해서 반환
        Parameter
            idx: int 조회할 index
        Return
            tuple: (torch.LongTensor, torch.FloatTensor) 
                    - x: 댓글 토큰_id 를 LongTensor(int64), y: 정답 Label을 FloatTensor(float32)
        """
        # self.texts : list[list]
        # self.labels: list[int]
        X = torch.tensor(self.texts[idx], dtype=torch.int64)
        y = torch.tensor(self.labels[idx], dtype=torch.float32).unsqueeze(dim=-1)

        return X, y
    

In [51]:
##### DataSet 생성
max_length = 30
trainset = NSMCDataset(train_inputs, train_labels, max_length, tokenizer)
testset = NSMCDataset(test_inputs, test_labels, max_length, tokenizer)

len(trainset), len(testset)

(150000, 50000)

In [54]:
# 테스트
c, l = trainset[3]
c, l

(tensor([11675,  5379,  5251,  8990,  5502,  5264,   920,  5258,   946,  5291,
          8201,     1,     1,     1,     1,     1,     1,     1,     1,     1,
             1,     1,     1,     1,     1,     1,     1,     1,     1,     1]),
 tensor([0.]))

In [56]:
tokenizer.decode(list(c))

'교도소 이야기 이다 구먼 솔직히 재미 는 없다 다 평점 조정'

In [57]:
######## DataLoader
batch_size = 100
train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(testset, batch_size=batch_size)

len(train_loader), len(test_loader)

(1500, 500)

# 모델링
- Embedding Layer를 이용해 Word Embedding Vector를 추출한다.
- LSTM을 이용해 Feature 추출
- Linear + Sigmoid로 댓글 긍정일 확률 출력
  
![outline](figures/rnn/RNN_outline.png)

## 모델 정의

In [ ]:
# 순전파: embedding layer -> LSTM Layer(문서-댓글에대한 feature) -> Linear+Sigmoid (분류기)

## 모델 생성

In [58]:
import torch
import torch.nn as nn

class NSMCClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, 
                       num_layers=1, bidirectional=True, dropout=0.2, pad_token_id=1):
        super().__init__()
        # EmbeddingLayer -> LSTM -> Linear -> Sigmoid

        # 토큰들(Tensor[int] - [20, 30, 1278, 30, ..], shape: [batch, max_length]) 
        #     -> embedding_layer -> 출력 tensor shape: [batch, seq_len, embedding_dim]
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size, # 어휘사전 크기
            embedding_dim=embedding_dim, # Embedding Vector의 차원수
            padding_idx=pad_token_id
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim, # 개별단어의 embedding 벡터 차원.
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            dropout=0.0 if num_layers==1 else dropout
        )

        # classifier의 입력 -> LSTM의 마지막 timestep의 hidden state
        #  out, (hidden, cell) = lstm(X)  -> out[-1]를 입력.
        self.classifier = nn.Linear(
            in_features= hidden_size * 2 if bidirectional else hidden_size, 
            out_features=1  # 이진분류 -> 양성일 확률값
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, X):
        # X: 입력 문서 - 토큰 리스트 [batch_size, seq_len(max_length)]
        # X -> Embedding  
        # 출력 shape: [batch_size, seq_len, embedding_dim]
        embedding_vector = self.embedding(X)

        # embedding vector의 batch 와 seq len index 위치를 변경.
        embedding_vector = embedding_vector.transpose(1, 0) # [seq_len, batch, em_dim]

        # embedding_vector -> LSTM
        out, _ = self.lstm(embedding_vector)
        # out shape: [seq_len, batch, hidden_size * bidirectional]

        # out[-1] (마지막 timestep) -> Classifier(Linear)
        output = self.classifier(out[-1])
        last_output = self.sigmoid(output)

        return last_output

In [59]:
# 하이퍼 파라미터/설정값 정의
device = "cuda" if torch.cuda.is_available() else "cpu"
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 100
hidden_size = 64
num_layers = 1
bidirectional = True

In [ ]:
model = NSMCClassifier(
    vocab_size=vocab_size, 
    embedding_dim=embedding_dim, 
    hidden_size=hidden_size,
    num_layers=num_layers,
    bidirectional=bidirectional
).to(device)

In [62]:
!uv pip install torchinfo

Checked 1 package in 5ms


In [ ]:
from torchinfo import summary
# summary(모델, 입력데이터shape) -> 입력데이터의 타입: float32
#   model의 입력 데이터 타입: int 64 (LongTensor) ==> 그래서 직접 int64 타입의 더미데이터를 만들어서 
#   넣어준다.
dummy_input = torch.randint(1, 10, (batch_size, max_length))
summary(model, input_data=dummy_input, device=device)

Layer (type:depth-idx)                   Output Shape              Param #
NSMCClassifier                           [100, 1]                  --
├─Embedding: 1-1                         [100, 30, 100]            2,485,500
├─LSTM: 1-2                              [30, 100, 128]            84,992
├─Linear: 1-3                            [100, 1]                  129
├─Sigmoid: 1-4                           [100, 1]                  --
Total params: 2,570,621
Trainable params: 2,570,621
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 503.54
Input size (MB): 0.02
Forward/backward pass size (MB): 5.47
Params size (MB): 10.28
Estimated Total Size (MB): 15.78

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
loss_fn = nn.BCELoss()

## 학습

### Train/Test 함수 정의

In [ ]:
def train(model, dataloader, loss_fn, optimizer, device="cpu"):
    # 1. 모델을 train모드
    model.train()
    # 2. Model을 device이동
    model = model.to(device)

    # 3. 1 에폭 학습
    train_loss = 0.0
    for X, y in dataloader:
        # X, y를 device로 이동
        X, y = X.to(device), y.to(device)
        # 추론
        pred = model(X)
        # loss계산
        loss = loss_fn(pred, y)
        # grad 계산
        loss.backward()
        # 파라미터 업데이트
        optimizer.step()
        # 파라미터 grad 초기화
        optimizer.zero_grad()
        # loss값 누적
        train_loss += loss.item()
    
    return train_loss / len(dataloader)

In [ ]:
# 평가함수
@torch.no_grad
def eval(model, dataloader, loss_fn, device="cpu"):
    # 모델 평가/검증시 사용
    #1. eval 모드로 변경
    model.eval()
    model = model.to(device)

    eval_loss, eval_acc = 0.0
    for X, y in dataloader:
        # X, y 를 이동
        X, y = X.to(device), y.to(device)
        # 추론
        pred_proba = model(X) # 양성일 확률
        pred_class = (pred_proba > 0.5).type(torch.int32) # True: 1, False: 0 => Label

        # 평가
        eval_loss += loss_fn(pred_proba, y).item()
        eval_acc += (pred_class == y).sum().item()

    return eval_loss / len(dataloader), eval_acc / len(dataloader.dataset)

### Train

## 모델저장

# 서비스

## 전처리 함수들

In [ ]:
from kiwipiepy import Kiwi
import string
import re

kiwi = Kiwi()
def text_preprocessing(text):
    """
    1. 영문 -> 소문자로 변환
    2. 구두점 제거
    3. 형태소 기반 토큰화
    4. 형태소로 토큰화 한 뒤 다시 하나의 문자열로 묶어서 반환.
    """
    text = text.lower()
    text = re.sub(rf"[{string.punctuation}]", ' ', text) # 구두점(특수문자)들을 ' '으로 변환.
    text = [token.lemma for token in kiwi.tokenize(text)] # [a, b, c,]
    return ' '.join(text)

In [ ]:
def pad_token_sequences(token_sequences, max_length):
    """padding 처리 메소드."""
    pad_token = tokenizer.token_to_id('<pad>')  
    seq_length = len(token_sequences)
    if seq_length > max_length:                 
        result = token_sequences[:max_length]
    else:                                            
        result = token_sequences + ([pad_token] * (max_length - seq_length))
    return result

In [ ]:
def predict_data_preprocessing(text_list):
    """
    모델에 입력할 수있는 input data를 생성
    Parameter:
        text_list: list - 추론할 댓글리스트
    Return
        torch.LongTensor - 댓글 token_id tensor
    """
   
    pass

## 추론

In [ ]:
comment_list = ["아 진짜 재미없다.", "여기 식당 먹을만 해요", "이걸 영화라고 만들었냐?", "기대 안하고 봐서 그런지 괜찮은데.", "이걸 영화라고 만들었나?", "아! 뭐야 진짜.", "재미있는데.", "연기 짱 좋아. 한번 더 볼 의향도 있다.", "뭐 그럭저럭"]